# Evaluate Performance Using Logistic Regression 

In [4]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from catboost import CatBoostRegressor

/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [6]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [7]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance


In [8]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

## **Data Preprocessing** 

In [9]:
preprocessor = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())

## **Feature Selection** 

In [10]:
# Conduct feature selection using shap_select.

def perform_feature_selection(X_train, y_train):
    results_dict = {}

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2)
    X_tr_fe = preprocessor.fit_transform(X_tr)
    X_te_fe = preprocessor.transform(X_te)
    X_te_df = pd.DataFrame(X_te_fe, columns=X_train.columns, index=y_te.index)
    
    
    for target in y_train.columns:
        model = XGBRegressor(n_estimators=1000, verbosity = 0, eval_metric='rmse', objective='reg:squarederror')
        model.fit(X_tr_fe, y_tr[target], eval_set=[(X_te_fe, y_te[target])])

        selected_df = shap_select(model, X_te_df, y_te[target], task="regression", threshold=0.05)
        results_dict[target] = selected_df

    return results_dict  

In [11]:
results = perform_feature_selection(X_train, y_train)

[0]	validation_0-rmse:61.14167
[1]	validation_0-rmse:52.73011
[2]	validation_0-rmse:47.30802
[3]	validation_0-rmse:42.90090
[4]	validation_0-rmse:40.19608
[5]	validation_0-rmse:38.74684
[6]	validation_0-rmse:37.21890
[7]	validation_0-rmse:36.75916
[8]	validation_0-rmse:36.00083
[9]	validation_0-rmse:35.53983
[10]	validation_0-rmse:35.12292
[11]	validation_0-rmse:34.86331
[12]	validation_0-rmse:34.67659
[13]	validation_0-rmse:34.62496
[14]	validation_0-rmse:33.96615
[15]	validation_0-rmse:33.91632
[16]	validation_0-rmse:33.72175
[17]	validation_0-rmse:33.64442
[18]	validation_0-rmse:33.52384
[19]	validation_0-rmse:33.33246
[20]	validation_0-rmse:33.26280
[21]	validation_0-rmse:33.32197
[22]	validation_0-rmse:33.23083
[23]	validation_0-rmse:33.19608
[24]	validation_0-rmse:33.26166
[25]	validation_0-rmse:33.17399
[26]	validation_0-rmse:33.11543
[27]	validation_0-rmse:33.11131
[28]	validation_0-rmse:32.98413
[29]	validation_0-rmse:32.93300
[30]	validation_0-rmse:32.97844
[31]	validation_0-

In [12]:
results

{'Total Alkalinity':                         feature name    t-value  stat.significance  \
 0                   skin_temperature  18.392507       2.369725e-67   
 1                 cec_pH_interaction  14.926905       1.246000e-46   
 2                          elevation  11.401734       9.020171e-29   
 3                  flow_accumulation   8.811196       3.882902e-18   
 4                                EVI   8.675614       1.207699e-17   
 5                             swir22   6.425324       1.842704e-10   
 6    evaporation_precipitation_ratio   6.357067       2.841427e-10   
 7                               clay   4.749550       2.265404e-06   
 8           Land Surface Temperature   3.039231       2.419335e-03   
 9                                 pH   2.424059       1.548454e-02   
 10                  soil_temperature   2.322332       2.036946e-02   
 11        phosphorous_pH_interaction   2.189053       2.877116e-02   
 12  flow_acc_phosphorous_interaction   1.859463       6.

In [13]:
alk_feats = [
    "skin_temperature",
    "cec_pH_interaction",
    "elevation",
    "flow_accumulation",
    "EVI",
    "swir22",
    "evaporation_precipitation_ratio",
    "clay",
    "Land Surface Temperature",
    "pH",
    "soil_temperature",
    "phosphorous_pH_interaction"
]

len(alk_feats)

12

In [14]:
elec_feats = [
    "pet",
    "NDVI_LST_interaction",
    "flow_acc_phosphorous_interaction",
    "evaporation_precipitation_ratio",
    "phosphorous_pH_interaction",
    "soil_temperature",
    "clay",
    "total_evaporation_sum",
    "phosphorous",
    "Land Surface Temperature",
    "elevation",
    "cec_pH_interaction"
]

len(elec_feats)

12

In [15]:
drp_feats = [
    "phosphorous_pH_interaction",
    "soil_temperature",
    "clay",
    "pet",
    "total_evaporation_sum",
    "pH",
    "NDVI_LST_interaction",
    "NDMI",
    "cec_pH_interaction",
    "nir",
    "elevation",
    "green"
]

len(drp_feats)

12

In [16]:
selected_feats = list(set(alk_feats + elec_feats + drp_feats))

len(selected_feats)

20